In [9]:
import json
from pathlib import Path
import pandas as pd
import re
from scripts import extract as ext

In [10]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [11]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/tipos_ejercicio.json", "r") as file:
    tipos_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [ ]:
def classify_exercise(exam_name: str, statement: str, exercise_types = tipos_ejercicio, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_year = int(exam_name.split("_")[0])
    exam_name = ", ".join(exam_name.split("_")[1:])

    # seleccionar modelo de examen en función del año
    if exam_year < 2020:
        exam_model = "modelo_2010_2019"
        exam_number_pattern = r"^\d{1}\.-"

    elif exam_year < 2024:
        exam_model = "modelo_2020_2023"
        exam_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2024+"
        exam_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    tipos_dict = tipos_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exam_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year < 2020:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in tipos_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif exam_year < 2024:  # 2020 - 2023 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2024 + - key 1, 2, 3, 4A, 4B, 5
            exam_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])



        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        pre_2020_exam_exercise_1_bool = exam_year < 2020 and exercise_number_key == "1"
        exam_2020_2024_period_exercise_1_bool =  (2020 <= exam_year < 2024) and exercise_number_key == "1"

        if pre_2020_exam_exercise_1_bool or exam_2020_2024_period_exercise_1_bool:
            exercise_topic = "Formulación"
            exercise_type = "Formulación"

        elif exam_year >= 2024 and exercise_number_key == "4A":
            exercise_topic = "Formulación"
            exercise_type = "Formulación orgánica"

        else:

            ###### CLASIFICACIÓN #######

            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = tipos_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topic = ""
            for topic in exercise_topic_options:
                
                # obtengo los tipos de ejercicios del tema y sus keywords
                topic_keywords = keywords[topic]["keywords_tema"]
                for kw in topic_keywords:
                    if kw in statement:
                        exercise_topic = topic
                        break

                # si se ha clasificado el ejercicio deja de buscar
                if exercise_topic:
                    break

            # lista de tipos de ejercicio
            exercise_types = []

            if exercise_topic:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                exercise_type_keywords = keywords[exercise_topic]["tipos"]
                for exercise_type, exercise_type_keywords in exercise_type_keywords.items():
                    # comprueba lista de keywords para ese tipo de ejercicio
                    # con que haga match una keyword con el enunciado
                    # clasifícalo com ese tipo de ejercicio
                    if any(kw in statement for kw in exercise_type_keywords):
                        exercise_types.append(exercise_type)
    
    return (exam_year, exam_name, exercise_number, exercise_topic, exercise_types)

In [13]:
# file_junio_2013 = ROOT / "examenes_quimica/2013_quimica/2013_quimica_junio_titular.pdf"

In [14]:
# junio_2013_exam_text = ext.extract_exercises_from_pdf(file_junio_2013)

# exam_year = int(file_junio_2013.name.split("_")[0])

# if exam_year < 2020:
#     year_period = 0
# elif exam_year < 2024:
#     year_period = 1
# else:
#     year_period = 2

# junio_2013_exercises_list = ext.split_exercises(junio_2013_exam_text, year_period=year_period)

In [15]:
# print(junio_2013_exercises_list[2])

In [16]:
# classify_exercise(file_junio_2023.stem, junio_2023_exercises_list[2], tipos_ejercicio, quimica_keywords)

In [19]:
_2013_classifications = []
for file in (ROOT / "examenes_quimica/2013_quimica").iterdir():
    if file.is_file():
        exam_text = ext.extract_exercises_from_pdf(file)
        exam_year = int(file.name.split("_")[0])

        if exam_year < 2020:
            year_period = 0
        elif exam_year < 2024:
            year_period = 1
        else:
            year_period = 2

        exercises_list = ext.split_exercises(exam_text, year_period=year_period)

        for ex in exercises_list:
            classifications = classify_exercise(file.stem, ex, tipos_ejercicio, quimica_keywords)
            _2013_classifications.append(classifications)

df = pd.DataFrame(_2013_classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type"])
df = df.explode("exercise_type", ignore_index=True)

In [20]:
df

,year,exam_name,exercise_number,topic,exercise_type
0,2013,"quimica, septiembre, titular",1,Formulación,modelo_2024+
1,2013,"quimica, septiembre, titular",1,Formulación,modelo_2020_2023
2,2013,"quimica, septiembre, titular",1,Formulación,modelo_2010_2019
3,2013,"quimica, septiembre, titular",2,enlace_quimico,Polaridad
4,2013,"quimica, septiembre, titular",3,redox,identificar oxidante y reductor
...,...,...,...,...,...
118,2013,"quimica, junio, titular",4,reactividad_organica,verdadero o falso
119,2013,"quimica, junio, titular",5,acido_base,disoluciones
120,2013,"quimica, junio, titular",5,acido_base,neutralización
121,2013,"quimica, junio, titular",6,equilibrio_quimico,Cálculo de Kc y Kp


In [21]:
df[df.topic == ""]

,year,exam_name,exercise_number,topic,exercise_type
6,2013,"quimica, septiembre, titular",5,,NaN
15,2013,"quimica, septiembre, titular",3,,NaN
29,2013,"quimica, septiembre, reserva, b",4,,NaN
35,2013,"quimica, septiembre, reserva, b",2,,NaN
48,2013,"quimica, junio, reserva, b",4,,NaN
59,2013,"quimica, junio, reserva, b",5,,NaN
71,2013,"quimica, septiembre, reserva, a",5,,NaN
76,2013,"quimica, septiembre, reserva, a",2,,NaN
80,2013,"quimica, septiembre, reserva, a",6,,NaN
91,2013,"quimica, junio, reserva, a",5,,NaN


In [23]:
df.shape

(123, 5)